<a href="https://colab.research.google.com/github/syedmahmoodiagents/Agents/blob/main/Langchain_LCEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import langchain, langchain_core

In [4]:
langchain.__version__

'1.2.0'

In [6]:
!pip install -U langchain_openai langchain_huggingface --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 23.1 MB/s eta 0:00:00


In [7]:
from langchain_openai import ChatOpenAI

In [8]:
import getpass, os

In [9]:
os.environ['OPENAI_API_KEY'] = getpass.getpass()

··········


In [10]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

In [49]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

In [12]:
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in one sentence, formatted as a JSON object with a single key named 'explanation'."
)

In [13]:
jsparser = JsonOutputParser()

In [24]:
# LCEL chain
chain = prompt | llm | jsparser

In [25]:
result = chain.invoke({"topic": "LCEL in LangChain"})

In [26]:
print(result)

{'explanation': 'LCEL (LangChain Event Log) is a feature in LangChain that enables the tracking and logging of events during the execution of language model chains, facilitating debugging and analysis.'}


# RunnableParallel

In [34]:
prompt2 = ChatPromptTemplate.from_template(
    "Explain {topic} in one sentence."
)

In [42]:
def clean_topic(x):
    return {"topic": x["topic"].strip().lower()}

In [43]:
preprocess = RunnableLambda(clean_topic)

In [20]:
parser = StrOutputParser()

In [45]:
chain2 = preprocess | prompt2 | llm | parser

In [46]:
chain2.invoke({"topic": "   LangChain LCEL   "})

"LangChain's LCEL (LangChain Embedding Language) is a framework designed to facilitate the integration and use of language models for various applications, enabling developers to create, manage, and deploy language-based workflows efficiently."

# RunnableParallel

In [47]:
summary_prompt = ChatPromptTemplate.from_template(
    "Give a 1-line summary of {topic}"
)

pros_prompt = ChatPromptTemplate.from_template(
    "List 3 advantages of {topic}"
)

In [48]:
summary_chain = summary_prompt | llm | parser
pros_chain = pros_prompt | llm | parser

In [50]:
parallel_chain = RunnableParallel(
    summary=summary_chain,
    pros=pros_chain
)

In [51]:
parallel_chain.invoke({"topic": "LangChain LCEL"})

{'summary': 'LangChain LCEL (Language Chain Enhanced Learning) is a framework designed to facilitate the development of applications that leverage language models for enhanced reasoning and decision-making capabilities.',
 'pros': "LangChain's LCEL (LangChain Embedding Language) offers several advantages for developers working with language models and embeddings. Here are three key advantages:\n\n1. **Modular Design**: LCEL promotes a modular approach to building applications with language models. This allows developers to easily swap out components, such as different embedding models or data sources, without needing to overhaul the entire system. This flexibility can lead to faster development cycles and easier experimentation with different configurations.\n\n2. **Enhanced Interoperability**: LCEL is designed to work seamlessly with various language models and embedding techniques. This interoperability enables developers to integrate multiple tools and libraries, making it easier to

# Routing

In [52]:
def route(x):
    if len(x["input"]) < 20:
        return "short"
    return "long"

In [53]:
router = RunnableLambda(route)

In [59]:
short_prompt = ChatPromptTemplate.from_template(
    "Give a short answer of which can be confined to one line: {input}"
)

long_prompt = ChatPromptTemplate.from_template(
    "Give a detailed explanation in as much as two lines: {input}"
)

In [60]:
routes = {
    "short": short_prompt | llm | parser,
    "long": long_prompt | llm | parser
}

In [61]:
chain = router | routes

In [62]:
chain.invoke({"input": "What is LCEL?"})

{'short': 'Concise and to the point.',
 'long': "Sure! Please specify the topic or subject you'd like a detailed explanation about, and I'll provide a concise response."}

# More Routing

In [23]:
from langchain_core.runnables import RunnableLambda

In [24]:
def route(x):
    text = x["input"].lower()
    if "summarize" in text:
        return "summary"
    if "translate" in text:
        return "translate"
    return "chat"


In [25]:
router = RunnableLambda(route)

In [26]:
summary_chain = ChatPromptTemplate.from_template(
    "Summarize: {input}"
) | llm | parser

translate_chain = ChatPromptTemplate.from_template(
    "Translate to French: {input}"
) | llm | parser

chat_chain = ChatPromptTemplate.from_template(
    "Chat normally: {input}"
) | llm | parser

In [27]:
routes = {
    "summary": summary_chain,
    "translate": translate_chain,
    "chat": chat_chain
}

In [28]:
chain = router | routes

In [29]:
chain.invoke({"input": "Summarize LCEL in one line"})

{'summary': 'A summary is a brief overview or condensed version of a larger text, capturing the main ideas and key points while omitting unnecessary details. It aims to provide a clear understanding of the original content in a shorter format.',
 'translate': 'The translation of "summary" in French is "résumé."',
 'chat': "Sure! How can I assist you today? If you have a specific topic or question in mind, feel free to share, and I'll provide a summary or information on that."}

# Streaming

In [64]:
for chunk in chain.stream({"input": "LCEL"}):
    print(chunk, end="", flush=True)

{'long': ''}{'long': 'Sure'}{'long': '!'}{'long': ' Please'}{'long': ' specify'}{'long': ' the'}{'long': ' topic'}{'long': ' or'}{'long': ' subject'}{'long': " you'd"}{'long': ' like'}{'long': ' a'}{'long': ' detailed'}{'long': ' explanation'}{'long': ' about'}{'long': ','}{'long': ' and'}{'long': " I'll"}{'long': ' provide'}{'long': ' a'}{'long': ' concise'}{'long': ' response'}{'long': '.'}{'long': ''}{'long': ''}{'long': ''}{'short': ''}{'short': 'Sure'}{'short': '!'}{'short': ' What'}{'short': ' would'}{'short': ' you'}{'short': ' like'}{'short': ' to'}{'short': ' know'}{'short': '?'}{'short': ''}{'short': ''}{'short': ''}

In [68]:
inputs = [
    {"input": "LCEL"},
    {"input": "LangGraph"},
    {"input": "RAG"},
]

In [69]:
chain.batch(inputs)

[{'short': 'Sure! What would you like to know?',
  'long': "Sure! Please specify the topic or subject you'd like a detailed explanation about, and I'll provide a concise response."},
 {'short': 'Sure! What would you like to know?',
  'long': "Sure! Please specify the topic or subject you'd like a detailed explanation about, and I'll provide a concise response."},
 {'short': 'Concise and to the point.',
  'long': "Sure! Please specify the topic or subject you'd like a detailed explanation about, and I'll provide a concise response."}]

# Hard Logic - As a function pipeline

In [14]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [15]:
think_prompt = ChatPromptTemplate.from_template(
    "Break down how to solve: {question}"
)

In [16]:
def compute(_):
    return {"result": 42}

In [17]:
compute_step = RunnableLambda(compute)

In [18]:
explain_prompt = ChatPromptTemplate.from_template(
    "Explain why the answer is {result}"
)

In [21]:
chain = think_prompt | llm | compute_step | explain_prompt | llm | parser

In [22]:
chain.invoke({"question": "What is the meaning of life?"})

'The answer "42" is famously known as the "Answer to the Ultimate Question of Life, the Universe, and Everything" from Douglas Adams\' science fiction series, "The Hitchhiker\'s Guide to the Galaxy." In the story, a group of hyper-intelligent beings builds a supercomputer named Deep Thought to calculate the answer to the ultimate question of existence. After seven and a half million years of computation, Deep Thought reveals that the answer is simply "42." \n\nHowever, the actual "Ultimate Question" itself is never revealed, leading to a humorous and philosophical exploration of the meaning of life and the absurdity of seeking a simple answer to complex questions. The choice of the number 42 has since become a cultural reference, symbolizing the search for meaning in a seemingly indifferent universe. \n\nIn summary, the answer is 42 because it represents a humorous and absurd conclusion to a profound question, highlighting the complexities of existence and the nature of inquiry.'

# InMemoryChatMessageHistory

In [43]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [44]:
history_store = {}

def get_history(session_id):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

In [45]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful tutor."),
    ("human", "{input}")
])

In [46]:
chain = prompt | llm

In [47]:
chat_chain = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input"
)

In [48]:
chat_chain.invoke(
    {"input": "What is LCEL?"},
    config={"configurable": {"session_id": "user1"}}
)

AIMessage(content='LCEL can refer to different things depending on the context, but one common interpretation is "Low-Cost Energy Lab" or "Low-Cost Energy Learning." It may also refer to specific organizations, programs, or technologies related to energy efficiency and sustainability.\n\nIf you have a specific context in mind, such as a particular field (like education, technology, or energy), please provide more details so I can give you a more accurate explanation!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 35, 'total_tokens': 123, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-CuCHqK74xeXIgBYq33EEUVo6B58up', 'service_tier': 'default',

# LCEL as a Router

In [49]:
def route(x):
    text = x["input"].lower()
    if "summarize" in text:
        return "summary"
    if "translate" in text:
        return "translate"
    return "chat"

In [50]:
router = RunnableLambda(route)

In [51]:
summary_chain = ChatPromptTemplate.from_template(
    "Summarize: {input}"
) | llm | parser

translate_chain = ChatPromptTemplate.from_template(
    "Translate to French: {input}"
) | llm | parser

chat_chain = ChatPromptTemplate.from_template(
    "Chat normally: {input}"
) | llm | parser

In [52]:
routes = {
    "summary": summary_chain,
    "translate": translate_chain,
    "chat": chat_chain
}

In [53]:
chain = router | routes

In [54]:
chain.invoke({"input": "Summarize LCEL in one line"})


{'summary': 'A summary is a brief overview or condensed version of a larger text, capturing the main ideas and key points while omitting unnecessary details. It aims to provide a clear understanding of the original content in a shorter format.',
 'translate': 'The translation of "summary" in French is "résumé."',
 'chat': "Sure! What would you like a summary of? Please provide some details or specify the topic you're interested in."}

# LCEL with RunnableParallel

In [78]:
critic = (
    ChatPromptTemplate.from_template("Criticize {topic}")
    | llm
    | parser
)

supporter = (
    ChatPromptTemplate.from_template("Support {topic}")
    | llm
    | parser
)

In [79]:
final_prompt = ChatPromptTemplate.from_template(
    """
    Topic: {topic}

    Criticism:
    {criticism}

    Support:
    {support}

    Give a balanced conclusion.
    """
)

In [80]:
chain = (
    RunnablePassthrough.assign(
        criticism=critic,
        support=supporter
    )
    | final_prompt
    | llm
    | parser
)

In [81]:
result = chain.invoke({"topic": "Using LCEL for production systems"})

In [82]:
result

'In conclusion, the use of LCEL (Logic-based Concurrent Event Language) in production systems presents both significant advantages and notable challenges. On one hand, LCEL can enhance agility, reduce development costs, and empower non-technical users to contribute to application development, thereby accelerating the response to changing business needs. Its ability to model concurrent events can streamline processes and improve overall system efficiency.\n\nHowever, the complexity of implementation, potential performance overhead, and challenges related to scalability and debugging cannot be overlooked. The learning curve associated with LCEL may hinder adoption among developers, and integration with existing systems can pose compatibility issues. Additionally, the risk of over-engineering and concurrency-related problems may complicate the development process.\n\nUltimately, organizations considering LCEL for their production systems should conduct a thorough assessment of their speci

# LCEL for RAG-like behavior

In [62]:
documents = {
    "lcel": "LCEL is LangChain Expression Language...",
    "langgraph": "LangGraph is a stateful agent framework..."
}

In [63]:
def retrieve(x):
    key = x["question"].lower()
    for k in documents:
        if k in key:
            return {"context": documents[k], "question": x["question"]}
    return {"context": "No context found", "question": x["question"]}

In [64]:
retriever = RunnableLambda(retrieve)

In [65]:
prompt = ChatPromptTemplate.from_template(
    """
    Context:
    {context}

    Question:
    {question}

    Answer concisely.
    """
)

In [66]:
chain = retriever | prompt | llm | parser

In [67]:
chain.invoke({"question": "What is LCEL?"})

'LCEL stands for LangChain Expression Language, which is a language designed for expressing and manipulating data within the LangChain framework.'

# LCEL as a Validator + Retry Loop

In [69]:
def validate(x):
    if "?" in x:
        return "good"
    return "bad"

In [70]:
validator = RunnableLambda(validate)

In [71]:
fix_prompt = ChatPromptTemplate.from_template(
    "Fix this answer to be more clear:\n{input}"
)

In [72]:
chain = (
    ChatPromptTemplate.from_template("Answer: {input}")
    | llm
    | parser
    | validator
    | {
        "good": RunnableLambda(lambda x: x),
        "bad": fix_prompt | llm | parser
    }
)

In [68]:
chain.invoke({"input": "LCEL"})

{'good': 'good',
 'bad': 'Sure! Could you please provide more context or specify what you would like to improve in the answer?'}

# LCEL for Batch Dataset Generation

In [73]:
questions = [
    {"topic": "LCEL"},
    {"topic": "LangGraph"},
    {"topic": "RAG"}
]

prompt = ChatPromptTemplate.from_template(
    "Generate one interview question about {topic}"
)

chain = prompt | llm | parser

chain.batch(questions)

['Certainly! Here’s an interview question related to LCEL (Language and Content in English Language Learning):\n\n"Can you explain how the integration of language and content in LCEL enhances the learning experience for students, and provide an example of a successful strategy you have used in your teaching practice?"',
 'What are the key features of LangGraph that differentiate it from other language processing frameworks, and how do these features enhance the development of natural language applications?',
 'Certainly! Here\'s an interview question about RAG (Retrieval-Augmented Generation):\n\n"Can you explain how Retrieval-Augmented Generation (RAG) improves the performance of language models in generating responses, and what are some potential challenges associated with integrating retrieval mechanisms into generative models?"']